In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv",encoding="latin-1")
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [3]:
from sklearn.model_selection import train_test_split

In [4]:
xtrain,xtest, ytrain, ytest = train_test_split(df["v2"], df["v1"], test_size=0.2, random_state=42)
# xtrain,xtest,ytrain,ytest

In [5]:
uq = set(xtrain.str.lower().str.findall(r"\b\w+\b").explode().dropna())

In [6]:
val,cnt = np.unique(xtrain.str.lower().str.findall(r"\b\w+\b").explode().dropna(), return_counts=True)
val,cnt

(array(['0', '00', '000', ..., 'ûïharry', 'ûò', 'ûówell'], dtype=object),
 array([ 3,  8, 23, ...,  1,  9,  1]))

In [7]:
vals,cnts = np.unique(xtrain.loc[ytrain=="spam"].str.lower().str.findall(r"\b\w+\b").explode().dropna(), return_counts=True)
vals,cnts

(array(['0', '00', '000', ..., 'ûªve', 'ûï', 'ûïharry'], dtype=object),
 array([ 3,  8, 23, ...,  1,  1,  1]))

In [8]:
#p(s|x)
psx = pd.DataFrame({"val":vals, "cnt":cnts})
psx["cnt"] = psx["cnt"]/ sum(psx["cnt"])
psx

,val,cnt
0,0,0.000194
1,00,0.000517
2,000,0.001487
3,008704050406,0.000129
4,0089,0.000065
...,...,...
2614,å,0.015191
2615,ì¼1,0.000065
2616,ûªve,0.000065
2617,ûï,0.000065


In [9]:
#p(s)
ps = sum(cnts)/sum(cnt)
ps

np.float64(0.21543860626401326)

In [10]:
#p(x)
px = pd.DataFrame({"val":val, "cnt":cnt})
px["cnt"] = px["cnt"]/sum(cnt)
px

,val,cnt
0,0,0.000042
1,00,0.000111
2,000,0.000320
3,000pes,0.000014
4,008704050406,0.000028
...,...,...
7770,ûªve,0.000014
7771,ûï,0.000014
7772,ûïharry,0.000014
7773,ûò,0.000125


In [11]:
px

,val,cnt
0,0,0.000042
1,00,0.000111
2,000,0.000320
3,000pes,0.000014
4,008704050406,0.000028
...,...,...
7770,ûªve,0.000014
7771,ûï,0.000014
7772,ûïharry,0.000014
7773,ûò,0.000125


In [12]:
pxs = ps*sum(np.log((ps* np.array([psx.set_index("val")["cnt"].get(x,0)/px.get(x) for x in xtest.iloc[0].split() if px.get(x) is not None and px.get(x) !=0]))))
pxs

np.float64(0.0)